# 02 · Phase 0: evidence for decisions D1, D4, D7

**Runtime: T4 GPU** (Runtime → Change runtime type → T4). Only the D4 section needs the GPU. D1 and D7 also run on CPU.

Produces the evidence the team needs to sign off the Phase 0 decisions in `docs/decisions.md`. Every output goes to `MyDrive/mpac/phase0/`.

| Section | Decision | Output on Drive |
|---|---|---|
| 1 | D1: dataset per objective | printed tables only |
| 2 | D4: MT model | `mt_round_trip.csv`, `mt_blind_<lang>.csv`, `mt_blind_key.csv` |
| 3 | D7: oracle sheet format | `oracle_demo/*.csv` |

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[dev,gpu]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
CONFIG = None   # team languages; use "configs/hi_pt.yaml" to compare MT on Hindi + Portuguese
from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.paths import subdir
cfg = load_config(*([CONFIG] if CONFIG else []))
TARGETS = cfg["languages"]["targets"]
OUT = subdir(cfg, "phase0")
print("targets:", TARGETS, "| outputs ->", OUT)

## 1 · D1: what the candidate datasets contain

PLURAL's preference data is 2 GB, so it is **streamed**: only a sample is read and nothing is downloaded in full.

In [ ]:
import pandas as pd
from datasets import load_dataset

# 1a. PLURAL survey questions -> forced-choice items (answer codes >= 0; negative codes are missing-data codes).
sq = load_dataset("agdhruv/plural-alignment", "survey_questions", split="train").to_pandas()
sq["n_options"] = sq["answer_options"].map(lambda opts: sum(int(o["code"]) >= 0 for o in opts))
print(len(sq), "survey questions; options per question:")
print(sq["n_options"].value_counts().sort_index().to_string())
sq[["question_code", "question_text", "n_options"]].head(10)

In [ ]:
# 1b. PLURAL preference pairs: a shuffled sample of 3,000 rows (rows on disk are grouped by country).
stream = load_dataset("agdhruv/plural-alignment", split="train", streaming=True).shuffle(seed=42, buffer_size=10_000)
plural = pd.DataFrame(list(stream.take(3000)))
plural["prompt_words"] = plural["prompt"].str.split().str.len()
plural["pref_words"] = plural["pref"].str.split().str.len()
print("countries in sample:", plural["country"].nunique())
print(plural[["prompt_words", "pref_words"]].describe().round(0).to_string())
print("\nexample prompt:\n", plural.loc[0, "prompt"])

In [ ]:
# 1c. Honesty and helpfulness candidates.
tqa = load_dataset("truthfulqa/truthful_qa", "multiple_choice", split="validation").to_pandas()
tqa["n_mc1"] = tqa["mc1_targets"].map(lambda t: len(t["choices"]))
print(f"TruthfulQA mc1: {len(tqa)} questions, options per question: {tqa['n_mc1'].min()}-{tqa['n_mc1'].max()}")

uf = pd.DataFrame(list(load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="test_prefs", streaming=True).take(2000)))
uf["prompt_words"] = uf["prompt"].str.split().str.len()
looks_like_code = uf["prompt"].str.contains(r"```|def |function|SELECT |#include|\bcode\b", regex=True)
usable = uf[(uf["prompt_words"] <= 120) & ~looks_like_code]
print(f"UltraFeedback test_prefs sample: {len(uf)} prompts, {len(usable)} short and non-code")
usable["prompt"].head(5).tolist()

## 2 · D4: compare the two MT candidates

60 English texts (30 PLURAL prompts and 30 TruthfulQA questions) are translated into each target and back. Round-trip chrF ranks the models. It favours literal translations, so the blind sheets for fluent speakers decide close calls.

Takes about 10–20 minutes on a T4. MADLAD runs in fp32 there because T5 models overflow in fp16.

In [ ]:
import gc, time, torch
from mpac.data.translate import MT_CANDIDATES, Translator, round_trip

texts = plural["prompt"].head(30).tolist() + tqa["question"].head(30).tolist()
results, forward = [], {}
for short, name in MT_CANDIDATES.items():
    t0 = time.time()
    mt = Translator(name)
    rt = round_trip(texts, mt, TARGETS)
    secs = time.time() - t0
    for lang, r in rt.items():
        results.append({"model": short, "lang": lang, "chrf": round(r["chrf"], 1)})
        forward[(short, lang)] = r["forward"]
    print(f"{short}: {secs/60:.1f} min")
    del mt; gc.collect(); torch.cuda.empty_cache()

table = pd.DataFrame(results).pivot(index="lang", columns="model", values="chrf")
table.loc["mean"] = table.mean()
table.to_csv(OUT / "mt_round_trip.csv")
table

In [ ]:
# Blind sheets: send mt_blind_<lang>.csv to a fluent speaker of <lang> (it opens in Google Sheets).
# They fill `better` with A, B or = . Don't share mt_blind_key.csv until the votes are in.
from mpac.data.translate import blind_comparison

keys = []
for lang in TARGETS:
    sheet, key = blind_comparison(texts, {m: forward[(m, lang)] for m in MT_CANDIDATES}, n=20)
    pd.DataFrame(sheet).to_csv(OUT / f"mt_blind_{lang}.csv", index=False)
    keys += [{"lang": lang, **k} for k in key]
pd.DataFrame(keys).to_csv(OUT / "mt_blind_key.csv", index=False)
print("wrote", sorted(p.name for p in OUT.glob("mt_blind_*.csv")))

In [ ]:
# After the votes come back: count wins per model and language.
def tally(lang):
    votes = pd.read_csv(OUT / f"mt_blind_{lang}.csv").merge(
        pd.read_csv(OUT / "mt_blind_key.csv").query("lang == @lang"), on="row", suffixes=("", "_model"))
    winner = votes.apply(lambda r: r["A_model"] if r["better"] == "A" else r["B_model"] if r["better"] == "B" else "tie", axis=1)
    return winner.value_counts()

# pd.DataFrame({lang: tally(lang) for lang in TARGETS}).fillna(0).astype(int)

## 3 · D7: what an annotator sheet looks like

Real sheets need real generations (Phase 1). This builds a demo from synthetic data so the team can review the format and `docs/oracle_protocol.md` before labelling starts.

In [ ]:
from mpac.data.oracle import export_sheets, sample_for_labelling
from mpac.data.splits import assign_splits
from mpac.data.synthetic import make_synthetic

items, gens, _ = make_synthetic(TARGETS, cfg["objectives"], n_items=60)
rows = sample_for_labelling(items, {("synthetic", "base"): gens}, n=30, targets=TARGETS,
                            splits=assign_splits(items, cfg["data"]["tune_frac"], cfg["project"]["seed"]))
paths = export_sheets(rows, OUT / "oracle_demo", ["annotator_a", "annotator_b"])
print(*paths, sep="\n")
pd.read_csv(paths[0]).head(3)

## Next

Record the outcome in `docs/decisions.md`: D1 sources, the D4 winner (with chrF and blind votes), and D7 sign-off. Also choose D8 (which PLURAL pairs train DPO) and D9 (language variants). Phase 1 (loaders and translation into the cache) starts from those choices.